# 04 – Evaluación de Modelos

Este notebook evalúa y compara los modelos entrenados:
- Carga de datos de prueba y modelos guardados
- Métricas de clasificación: exactitud, precisión, recall, F1, ROC-AUC
- Matrices de confusión
- Gráfico comparativo de métricas
- Identificación del mejor modelo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.insert(0, os.path.abspath('..'))
from src.models import cargar_modelo
from src.evaluation import (
    evaluar_clasificacion,
    graficar_matriz_confusion,
    graficar_comparacion_modelos,
)

sns.set_theme(style='whitegrid')

## 1. Carga de datos de prueba

In [ ]:
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f'Conjunto de prueba: {X_test.shape}')

## 2. Carga de modelos entrenados

In [ ]:
directorio_modelos = '../models'
modelos_entrenados = {}

for archivo in os.listdir(directorio_modelos):
    if archivo.endswith('.joblib'):
        nombre = archivo.replace('.joblib', '').replace('_', ' ').title()
        ruta = os.path.join(directorio_modelos, archivo)
        modelos_entrenados[nombre] = cargar_modelo(ruta)
        print(f'Cargado: {nombre}')

print(f'\nTotal de modelos cargados: {len(modelos_entrenados)}')

## 3. Métricas de evaluación

In [ ]:
df_resultados = evaluar_clasificacion(modelos_entrenados, X_test, y_test)
df_resultados

## 4. Gráfico comparativo

In [ ]:
graficar_comparacion_modelos(
    df_resultados,
    metrica='F1-Score',
    titulo='Comparación de Modelos — F1-Score',
    directorio='../reports/figures',
)

## 5. Matriz de confusión del mejor modelo

In [ ]:
mejor_nombre = df_resultados.iloc[0]['Modelo']
mejor_modelo = modelos_entrenados[mejor_nombre]

print(f'Mejor modelo: {mejor_nombre}')
graficar_matriz_confusion(
    mejor_modelo, X_test, y_test,
    nombre_modelo=mejor_nombre,
    directorio='../reports/figures',
)

## 6. Reporte de clasificación detallado del mejor modelo

In [ ]:
from sklearn.metrics import classification_report

y_pred = mejor_modelo.predict(X_test)
print(f'=== Reporte: {mejor_nombre} ===')
print(classification_report(y_test, y_pred))